# 04 Buyback Announcement Classification with Ollama + Qwen3 14B

This notebook classifies buyback-related excerpts in `data/buyback_llm_windows.csv` into:

- `ANNOUNCEMENT`
- `NOT_ANNOUNCEMENT`

The workflow is intentionally validation-first:

1. Explore the input file and confirm schema.
2. Verify Ollama and `qwen3:14b` are available locally.
3. Run a 100-row validation sample and save it for manual review.
4. Only after review, enable the full checkpointed run.


In [25]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT
elif not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_PATH = PROJECT_ROOT / 'data' / 'buyback_llm_windows.csv'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = 'http://localhost:11434/api/generate'
MODEL = 'qwen3:14b'
KEEP_ALIVE = '24h'
MAX_WORKERS = 8

TEXT_COL = 'relevant_text_only'
ID_COLS = ['companyid', 'keydevid', 'announceddate', 'mostimportantdateutc']

VALIDATION_OUTPUT = INTERIM_DIR / 'validation_sample_100.csv'
CHECKPOINT_PATH = INTERIM_DIR / 'classification_checkpoint.csv'
FULL_OUTPUT_PATH = INTERIM_DIR / 'transcripts_classified_full.csv'
ANNOUNCEMENTS_ONLY_PATH = INTERIM_DIR / 'buyback_announcements_only.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Input path: {INPUT_PATH}')
print(f'Input exists: {INPUT_PATH.exists()}')





Project root: C:\Users\javed\Documents\Projects\Analytical Finance and Machine Learning
Input path: C:\Users\javed\Documents\Projects\Analytical Finance and Machine Learning\data\buyback_llm_windows.csv
Input exists: True


## 1. Explore the input file

This dataset already has the fields we need for classification:

- text: `relevant_text_only`
- IDs: `companyid`, `keydevid`
- dates: `announceddate`, `mostimportantdateutc`


In [ ]:
df = pd.read_csv(INPUT_PATH)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(df.head(2).to_string())
print(df.dtypes.to_string())

print('\nText length summary:')
print(df[TEXT_COL].dropna().astype(str).str.len().describe())


Shape: (33875, 5)
Columns: ['companyid', 'keydevid', 'announceddate', 'mostimportantdateutc', 'relevant_text_only']
   companyid    keydevid        announceddate mostimportantdateutc                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

## 2. Install and verify Ollama + Qwen3 14B

Run these in a terminal if needed:

```bash
ollama --version
ollama pull qwen3:14b
ollama run qwen3:14b "/nothink Say hello" --verbose
```

If Ollama is not installed:

- Windows: https://ollama.com/download
- Linux: `curl -fsSL https://ollama.com/install.sh | sh`

If the service is not running, start it with `ollama serve`.


In [ ]:
def run_cmd(command: list[str]) -> None:
    print(f"$ {' '.join(command)}")
    try:
        completed = subprocess.run(command, check=False, capture_output=True, text=True)
        print(completed.stdout.strip() or '(no stdout)')
        if completed.stderr.strip():
            print('\n[stderr]')
            print(completed.stderr.strip())
        print(f'Exit code: {completed.returncode}')
    except FileNotFoundError:
        print('Command not found on PATH.')


ollama_path = shutil.which('ollama')
print(f'Ollama binary: {ollama_path}')

if ollama_path:
    run_cmd([ollama_path, '--version'])
    run_cmd([ollama_path, 'list'])
else:
    print('Ollama is not installed or not on PATH in this environment.')

try:
    resp = requests.post(
        OLLAMA_URL,
        json={
            'model': MODEL,
            'prompt': '/nothink Say hello',
            'stream': False,
            'options': {'num_predict': 10},
        },
        timeout=600,
    )
    print(f'API status: {resp.status_code}')
    print(resp.json().get('response', ''))
    print('Ollama + Qwen3 14B is ready.')
except Exception as exc:
    print(f'Ollama API check failed: {type(exc).__name__}: {exc}')
    print('Ollama is installed, but the local API request did not complete successfully. Warm up the model and retry.')


Ollama binary: C:\Users\javed\AppData\Local\Programs\Ollama\ollama.EXE
$ C:\Users\javed\AppData\Local\Programs\Ollama\ollama.EXE --version
ollama version is 0.20.5
Exit code: 0
$ C:\Users\javed\AppData\Local\Programs\Ollama\ollama.EXE list
NAME                            ID              SIZE      MODIFIED     
qwen3:14b                       bdbd181c33f2    9.3 GB    5 hours ago     
gemini-3-flash-preview:cloud    ebade0d31690    -         3 months ago    
mistral:latest                  6577803aa9a0    4.4 GB    3 months ago
Exit code: 0
API status: 200

Ollama + Qwen3 14B is ready.


## 3. Build the classification function

The prompt is tuned to distinguish new or expanded authorizations from discussion of existing programs.


In [ ]:
PROMPT_TEMPLATE = """/nothink
Classify this earnings-call excerpt about share repurchases.

Return exactly one label and nothing else:
ANNOUNCEMENT
or
NOT_ANNOUNCEMENT

Use ANNOUNCEMENT only if the excerpt clearly says a new buyback authorization was approved, or an existing authorization was expanded, increased, renewed, or extended.

Use NOT_ANNOUNCEMENT for discussion of an existing program, past repurchases, vague buyback intentions, analyst questions without a concrete authorization, or generic capital allocation commentary.

Also use NOT_ANNOUNCEMENT if the excerpt says the authorization was announced previously, for example phrases like 'as announced last quarter,' 'as you recall in our last call,' 'we announced in our last call,' 'previously disclosed,' 'earlier announced,' or 'remaining under the current authorization.'
A sentence such as 'as you recall in our last call, we announced an additional authorization' is NOT_ANNOUNCEMENT because the announcement happened earlier, not in the current excerpt.

If ambiguous, return NOT_ANNOUNCEMENT.

Excerpt:
---
{text}
---
"""

FALLBACK_PROMPT_TEMPLATE = """Decide whether this excerpt announces a new or expanded share repurchase authorization.
Answer with one label only: ANNOUNCEMENT or NOT_ANNOUNCEMENT.
If uncertain, answer NOT_ANNOUNCEMENT.
If the text refers to a prior or previously announced authorization rather than a new authorization in this excerpt, answer NOT_ANNOUNCEMENT.
If the text says the company is merely executing under a prior authorization, answer NOT_ANNOUNCEMENT.

Excerpt:
{text}
"""


def parse_label_from_text(raw_text: str) -> tuple[str | None, str]:
    text = (raw_text or '').strip()
    if not text:
        return None, text

    cleaned = text.replace('"', ' ').replace("'", ' ').replace('`', ' ')
    cleaned = cleaned.replace('{', ' ').replace('}', ' ').replace(':', ' ')
    cleaned = ' '.join(cleaned.split())
    upper = cleaned.upper()

    if 'NOT_ANNOUNCEMENT' in upper or 'NOT ANNOUNCEMENT' in upper:
        return 'NOT_ANNOUNCEMENT', text
    if upper.startswith('ANNOUNCEMENT') or ' ANNOUNCEMENT ' in f' {upper} ':
        return 'ANNOUNCEMENT', text

    return None, text


def _call_ollama(prompt: str, num_predict: int = 8):
    return requests.post(
        OLLAMA_URL,
        json={
            'model': MODEL,
            'prompt': prompt,
            'stream': False,
            'think': False,
            'keep_alive': KEEP_ALIVE,
            'options': {
                'temperature': 0.0,
                'num_predict': num_predict,
            },
        },
        timeout=600,
    )


def classify_buyback(text: object, max_chars: int = 2500, return_raw: bool = False):
    if pd.isna(text) or not isinstance(text, str) or len(text.strip()) < 10:
        return {'label': 'SKIP_EMPTY', 'raw_response': ''} if return_raw else 'SKIP_EMPTY'

    truncated = text[:max_chars]

    try:
        prompts = [
            PROMPT_TEMPLATE.format(text=truncated),
            FALLBACK_PROMPT_TEMPLATE.format(text=truncated),
        ]
        raw_responses = []
        final_payload = None
        final_label = None

        for attempt_idx, prompt in enumerate(prompts, start=1):
            resp = _call_ollama(prompt, num_predict=8)
            if resp.status_code != 200:
                label = f"HTTP_ERROR:{resp.status_code}:{resp.text[:200]}"
                return {'label': label, 'raw_response': resp.text[:500]} if return_raw else label

            payload = resp.json()
            raw_response = (payload.get('response') or '').strip()
            thinking = (payload.get('thinking') or '').strip()
            raw_responses.append(f"attempt_{attempt_idx}: response={raw_response} | thinking={thinking[:200]}")
            label, _ = parse_label_from_text(raw_response)
            final_payload = payload
            if label is not None:
                final_label = label
                break

        if final_label is None:
            final_label = f"UNCLEAR:{raw_responses[-1][:80] if raw_responses else ''}"

        if return_raw:
            return {
                'label': final_label,
                'raw_response': ' || '.join(raw_responses),
                'done_reason': final_payload.get('done_reason') if final_payload else None,
                'thinking': final_payload.get('thinking') if final_payload else None,
                'eval_count': final_payload.get('eval_count') if final_payload else None,
                'prompt_eval_count': final_payload.get('prompt_eval_count') if final_payload else None,
            }
        return final_label

    except requests.exceptions.Timeout:
        return {'label': 'TIMEOUT', 'raw_response': ''} if return_raw else 'TIMEOUT'
    except requests.exceptions.ConnectionError:
        return {'label': 'CONNECTION_ERROR', 'raw_response': ''} if return_raw else 'CONNECTION_ERROR'
    except Exception as exc:
        label = f"ERROR:{str(exc)[:80]}"
        return {'label': label, 'raw_response': ''} if return_raw else label



## 3b. Warm up Ollama before validation\n
\n
The first request to a 14B model can take a while. Run this once before the 100-row validation so you do not burn the sample on cold-start timeouts.\n

In [29]:
print('Warming up Ollama...')
warmup = requests.post(
    OLLAMA_URL,
    json={
        'model': MODEL,
        'prompt': '/nothink Say hello',
        'stream': False,
        'think': False,
        'keep_alive': KEEP_ALIVE,
        'options': {
            'temperature': 0.0,
            'num_predict': 10,
        },
    },
    timeout=600,
)
print(f'Status: {warmup.status_code}')
if warmup.status_code == 200:
    warmup_payload = warmup.json()
    print(warmup_payload.get('response', ''))
    print(f"Warmup complete. load_duration_ns={warmup_payload.get('load_duration')} total_duration_ns={warmup_payload.get('total_duration')}")
else:
    print(warmup.text[:500])
    print('Warmup did not return HTTP 200. Fix this before validation.')



Warming up Ollama...
Status: 200
Hello! How can I assist you today? 
Warmup complete. load_duration_ns=61963800 total_duration_ns=200463000


## 3c. Debug a few raw model responses\n
Run this before the 100-row validation if labels still look wrong.\n

In [30]:
debug_df = pd.read_csv(INPUT_PATH)
debug_sample = debug_df.sample(n=min(5, len(debug_df)), random_state=7).copy()

debug_records = []
for _, row in debug_sample.iterrows():
    result = classify_buyback(row[TEXT_COL], return_raw=True)
    debug_records.append({
        **{col: row[col] for col in ID_COLS if col in row.index},
        TEXT_COL: row[TEXT_COL],
        'classification': result['label'],
        'raw_response': result.get('raw_response', ''),
        'done_reason': result.get('done_reason', ''),
        'eval_count': result.get('eval_count', ''),
        'thinking': result.get('thinking', ''),
    })

debug_results_df = pd.DataFrame(debug_records)
display(debug_results_df[[c for c in ID_COLS + ['classification', 'done_reason', 'eval_count'] if c in debug_results_df.columns]])
print('Raw response samples:')
for i, row in enumerate(debug_results_df[['raw_response', 'thinking']].to_dict('records'), start=1):
    print(f'\nSample {i} raw: {row["raw_response"][:500]}')
    print(f'Sample {i} thinking: {str(row["thinking"])[:500]}')



,companyid,keydevid,announceddate,mostimportantdateutc,classification,done_reason,eval_count
0,250178,6.537983e+08,2020-02-04 21:26:00,2020-02-05,NOT_ANNOUNCEMENT,stop,6
1,24734352,1.903429e+09,2024-11-07 00:00:00,2024-11-07,NOT_ANNOUNCEMENT,stop,6
2,372526,5.891692e+08,2018-10-24 10:08:00,2018-11-01,ANNOUNCEMENT,stop,4
3,260005529,4.179834e+08,2017-01-24 16:20:00,2017-01-25,ANNOUNCEMENT,stop,4
4,296308,6.313678e+08,2019-07-29 00:00:00,2019-07-30,ANNOUNCEMENT,stop,4


Raw response samples:

Sample 1 raw: attempt_1: response=NOT_ANNOUNCEMENT | thinking=
Sample 1 thinking: None

Sample 2 raw: attempt_1: response=NOT_ANNOUNCEMENT | thinking=
Sample 2 thinking: None

Sample 3 raw: attempt_1: response=ANNOUNCEMENT | thinking=
Sample 3 thinking: None

Sample 4 raw: attempt_1: response=ANNOUNCEMENT | thinking=
Sample 4 thinking: None

Sample 5 raw: attempt_1: response=ANNOUNCEMENT | thinking=
Sample 5 thinking: None


## 4. Validation run on 100 rows

Do this first. Review the saved CSV manually before enabling the full batch.


In [31]:
validation_df = pd.read_csv(INPUT_PATH)

print(f'Total rows: {len(validation_df)}')
print(f'Text column: {TEXT_COL}')
print(f'Max workers: {MAX_WORKERS}')
print(validation_df[TEXT_COL].dropna().astype(str).str.len().describe())

sample = validation_df.sample(n=min(100, len(validation_df)), random_state=42).copy().reset_index(drop=True)

print('\nRunning validation on 100 transcripts...')
start = time.time()

texts = sample[TEXT_COL].tolist()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    results = list(tqdm(executor.map(lambda text: classify_buyback(text, return_raw=True), texts), total=len(texts), desc='Validating'))

sample['classification'] = [result['label'] for result in results]
sample['raw_response'] = [result.get('raw_response', '') for result in results]
sample['thinking'] = [str(result.get('thinking', '')) for result in results]
elapsed = time.time() - start

print(f'\nValidation complete in {elapsed:.1f} seconds ({len(sample) / elapsed:.2f} transcripts/sec)')
print('\nResults:')
print(sample['classification'].value_counts())

rate = len(sample) / elapsed if elapsed > 0 else float('nan')
full_time_hours = len(validation_df) / rate / 3600 if rate > 0 else float('nan')
print(f'\nEstimated full run time: {full_time_hours:.1f} hours')

review_cols = [c for c in ID_COLS + [TEXT_COL, 'classification', 'raw_response', 'thinking'] if c in sample.columns]
sample[review_cols].to_csv(VALIDATION_OUTPUT, index=False)

print(f'\nSaved validation sample to {VALIDATION_OUTPUT}')
print('>>> MANUALLY REVIEW THIS FILE before running the full batch <<<')
print('>>> Check 20-30 rows to verify classification accuracy <<<')



Total rows: 33875
Text column: relevant_text_only
Max workers: 8
count    33875.000000
mean      3137.800886
std       2198.382429
min        768.000000
25%       1332.000000
50%       2518.000000
75%       4123.000000
max      45461.000000
Name: relevant_text_only, dtype: float64

Running validation on 100 transcripts...


Validating:   0%|          | 0/100 [00:00<?, ?it/s]


Validation complete in 30.8 seconds (3.25 transcripts/sec)

Results:
classification
NOT_ANNOUNCEMENT    72
ANNOUNCEMENT        28
Name: count, dtype: int64

Estimated full run time: 2.9 hours

Saved validation sample to C:\Users\javed\Documents\Projects\Analytical Finance and Machine Learning\data\interim\validation_sample_100.csv
>>> MANUALLY REVIEW THIS FILE before running the full batch <<<
>>> Check 20-30 rows to verify classification accuracy <<<


## STOP HERE

Review `data/interim/validation_sample_100.csv` before running the full batch.

Common failure mode to check for:

- model calls an existing-program discussion an `ANNOUNCEMENT`

Only continue after tightening the prompt if needed.


## 5. Full classification run with checkpointing
Enable this only after validation looks clean.


In [32]:
RUN_FULL_BATCH = True

if not RUN_FULL_BATCH:
    raise RuntimeError(
        'Validation gate is active. Review data/interim/validation_sample_100.csv first, then set RUN_FULL_BATCH = True.'
    )

full_df = pd.read_csv(INPUT_PATH)
start_idx = 0

if CHECKPOINT_PATH.exists():
    checkpoint = pd.read_csv(CHECKPOINT_PATH)
    start_idx = len(checkpoint)
    results = checkpoint['classification'].tolist()
    print(f'Resuming from checkpoint at row {start_idx}/{len(full_df)}')
else:
    results = []
    print(f'Starting fresh classification of {len(full_df)} transcripts')

print(f'Model: {MODEL}')
print(f'Text column: {TEXT_COL}')
print(f'Max workers: {MAX_WORKERS}')
print(f'Checkpoint path: {CHECKPOINT_PATH}')
start = time.time()
CHECKPOINT_INTERVAL = 500

for chunk_start in tqdm(range(start_idx, len(full_df), CHECKPOINT_INTERVAL), initial=start_idx, total=len(full_df), desc='Classifying'):
    chunk_df = full_df.iloc[chunk_start: chunk_start + CHECKPOINT_INTERVAL].copy()
    texts = chunk_df[TEXT_COL].tolist()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        chunk_results = list(executor.map(classify_buyback, texts))
    results.extend(chunk_results)

    processed = len(results)
    checkpoint_df = full_df.iloc[:processed].copy()
    checkpoint_df['classification'] = results
    checkpoint_df.to_csv(CHECKPOINT_PATH, index=False)

    elapsed = time.time() - start
    rate = (processed - start_idx) / elapsed if elapsed > 0 else 0.0
    remaining = (len(full_df) - processed) / rate if rate > 0 else 0.0
    counts = pd.Series(results).value_counts()

    print(f'\nCheckpoint saved at {processed}/{len(full_df)}')
    print(f'Rate: {rate:.2f}/sec | ETA: {remaining / 3600:.1f}hr')
    print(
        f"ANNOUNCEMENT: {counts.get('ANNOUNCEMENT', 0)} | "
        f"NOT_ANNOUNCEMENT: {counts.get('NOT_ANNOUNCEMENT', 0)} | "
        f"Other: {len(results) - counts.get('ANNOUNCEMENT', 0) - counts.get('NOT_ANNOUNCEMENT', 0)}"
    )

full_df['classification'] = results
total_time = time.time() - start

print('\n' + '=' * 60)
print('CLASSIFICATION COMPLETE')
print('=' * 60)
print(f'Total time: {total_time / 3600:.2f} hours')
print(f'Total transcripts: {len(full_df)}')
print('\nResults:')
print(full_df['classification'].value_counts())
print(
    f"\nError/unclear rate: "
    f"{(~full_df['classification'].isin(['ANNOUNCEMENT', 'NOT_ANNOUNCEMENT'])).mean() * 100:.1f}%"
)

full_df.to_csv(FULL_OUTPUT_PATH, index=False)
print(f'\nSaved: {FULL_OUTPUT_PATH}')

announcements = full_df.loc[full_df['classification'] == 'ANNOUNCEMENT'].copy()
print(f'\nAnnouncement transcripts: {len(announcements)} ({100 * len(announcements) / len(full_df):.1f}%)')
announcements.to_csv(ANNOUNCEMENTS_ONLY_PATH, index=False)
print(f'Saved: {ANNOUNCEMENTS_ONLY_PATH}')

if CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print('Cleaned up checkpoint file.')



Starting fresh classification of 33875 transcripts
Model: qwen3:14b
Text column: relevant_text_only
Max workers: 8
Checkpoint path: C:\Users\javed\Documents\Projects\Analytical Finance and Machine Learning\data\interim\classification_checkpoint.csv


Classifying:   0%|          | 0/33875 [00:00<?, ?it/s]


Checkpoint saved at 500/33875
Rate: 3.42/sec | ETA: 2.7hr
ANNOUNCEMENT: 146 | NOT_ANNOUNCEMENT: 354 | Other: 0

Checkpoint saved at 1000/33875
Rate: 3.37/sec | ETA: 2.7hr
ANNOUNCEMENT: 297 | NOT_ANNOUNCEMENT: 703 | Other: 0

Checkpoint saved at 1500/33875
Rate: 3.37/sec | ETA: 2.7hr
ANNOUNCEMENT: 451 | NOT_ANNOUNCEMENT: 1049 | Other: 0

Checkpoint saved at 2000/33875
Rate: 3.37/sec | ETA: 2.6hr
ANNOUNCEMENT: 592 | NOT_ANNOUNCEMENT: 1408 | Other: 0

Checkpoint saved at 2500/33875
Rate: 3.38/sec | ETA: 2.6hr
ANNOUNCEMENT: 708 | NOT_ANNOUNCEMENT: 1792 | Other: 0

Checkpoint saved at 3000/33875
Rate: 3.38/sec | ETA: 2.5hr
ANNOUNCEMENT: 819 | NOT_ANNOUNCEMENT: 2181 | Other: 0

Checkpoint saved at 3500/33875
Rate: 3.38/sec | ETA: 2.5hr
ANNOUNCEMENT: 935 | NOT_ANNOUNCEMENT: 2565 | Other: 0

Checkpoint saved at 4000/33875
Rate: 3.39/sec | ETA: 2.5hr
ANNOUNCEMENT: 1050 | NOT_ANNOUNCEMENT: 2950 | Other: 0

Checkpoint saved at 4500/33875
Rate: 3.39/sec | ETA: 2.4hr
ANNOUNCEMENT: 1185 | NOT_ANNOU